In [ ]:
# %pip install langchain langchain-groq pinecone-client sentence-transformers pandas unstructured

  Obtaining dependency information for numpy<2 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.2
    Uninstalling numpy-2.3.2:
      Successfully uninstalled numpy-2.3.2
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Shrirang\\AppData\\Roaming\\Python\\Python311\\site-packages\\~umpy.libs\\libscipy_openblas64_-860d95b1c38e637ce4509f5fa24fbf2a.dll'
Consider using the `--user` option or check the permissions.



In [ ]:
# Fix transformers compatibility with NumPy 1.x
%pip install --upgrade transformers sentence-transformers

In [3]:
# %pip install langchain_community langchain_pinecone langchain_groq langchain_core

In [4]:
import os
from getpass import getpass

# 2. Set up API keys (using environment variables or getpass for security)
# os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API key: ")
# os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API key: ")

# For now, using hardcoded keys (remove these in production)
os.environ["GROQ_API_KEY"] = "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"
os.environ["PINECONE_API_KEY"] = "pcsk_6kCUw1_8H1Qg3znYCht8vR7MXdFb2XeXb4X3mGXoV5K98xu69b3PmDJBf5BBdaf9PkXnRn"

from langchain.document_loaders import DirectoryLoader, UnstructuredFileLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
# Removed duplicate import: from langchain.schema import Document

# This helper function processes a single CSV file row by row.
def create_documents_from_csv(filepath):
    """
    Reads a CSV file and creates a LangChain Document for each row.
    Each document's page_content is a string representation of the row.
    """
    try:
        df = pd.read_csv(filepath)
        documents = []
        for index, row in df.iterrows():
            # Create a descriptive string for the row's content
            # This combines column names with their values for context
            content = ", ".join([f"{col}: {val}" for col, val in row.astype(str).items()])

            # Create a Document for each row
            doc = Document(
                page_content=content,
                metadata={
                    "source": filepath,
                    "row_index": index  # Store original row number for reference
                }
            )
            documents.append(doc)
        return documents
    except Exception as e:
        print(f"Error processing CSV file {os.path.basename(filepath)}: {e}")
        return []

# Your main function, now updated to use the helper function.
def get_documents_from_directory(directory_path):
    """
    Loads documents from a directory. CSVs are processed row-by-row for context.
    Other files are handled by UnstructuredFileLoader.
    """
    all_documents = []
    for filename in os.listdir(directory_path):
        filepath = os.path.join(directory_path, filename)
        if os.path.isfile(filepath):
            print(f"Processing file: {filename}")
            try:
                if filename.lower().endswith('.csv'):
                    # Use our new function to get one document per row
                    csv_docs = create_documents_from_csv(filepath)
                    all_documents.extend(csv_docs)
                else:
                    # Keep using UnstructuredFileLoader for other types
                    # Note: You might need to install 'unstructured'
                    from langchain_community.document_loaders import UnstructuredFileLoader
                    loader = UnstructuredFileLoader(filepath)
                    all_documents.extend(loader.load())
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
    return all_documents



# 4. Embedding Model
def get_embedding_model():
    """
    Loads the BAAI embedding model from Hugging Face.
    """
    model_name = "BAAI/bge-large-en-v1.5"
    model_kwargs = {"device": "cpu"}
    encode_kwargs = {"normalize_embeddings": True}
    return HuggingFaceBgeEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )

# 5. Pinecone Setup
def setup_pinecone(index_name):
    """
    Initializes Pinecone and creates an index if it doesn't exist.
    """
    from pinecone import Pinecone, ServerlessSpec

    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

    if index_name not in pc.list_indexes().names():
        print(f"Creating new Pinecone index: {index_name}")
        pc.create_index(
            name=index_name,
            dimension=1024,  # Dimension of the BAAI model
            metric="cosine",
            spec=ServerlessSpec(
                cloud='aws',
                region='us-east-1'
            )
        )
    return pc.Index(index_name)

In [5]:
# 6. RAG Chain Invocation
def get_rag_chain(vectorstore):
    """
    Creates and returns the RAG chain.
    """
    llm = ChatGroq(model_name="llama3-8b-8192", temperature=0)

    system_prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the "
        "answer concise."
        "\n\n"
        "{context}"
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}"),
        ]
    )

    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(vectorstore.as_retriever(), question_answer_chain)
    return rag_chain

In [9]:
# 7. Main function
import pandas as pd
def main():
    # Configuration
    PINECONE_INDEX_NAME = "capitol-data-index"
    # Create a directory named 'docs' and place your files there
    DIRECTORY_PATH = "./"

    if not os.path.exists(DIRECTORY_PATH):
        os.makedirs(DIRECTORY_PATH)
        print(f"Created directory '{DIRECTORY_PATH}'. Please add your documents to this folder and run again.")
        return

    # Load documents from the directory
    print(f"Loading documents from '{DIRECTORY_PATH}'...")
    documents = get_documents_from_directory(DIRECTORY_PATH)

    if not documents:
        print(f"No documents found in '{DIRECTORY_PATH}'. Please add files to process.")
        return

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = text_splitter.split_documents(documents)

    # Get embedding model
    print("Loading embedding model...")
    embedding_model = get_embedding_model()

    # Initialize Pinecone
    setup_pinecone(PINECONE_INDEX_NAME)

    # Create and store embeddings in Pinecone
    print("Creating and storing embeddings in Pinecone...")
    vectorstore = PineconeVectorStore.from_documents(docs, embedding_model, index_name=PINECONE_INDEX_NAME)

    # Set up the RAG chain
    print("Setting up the RAG chain...")
    rag_chain = get_rag_chain(vectorstore)

    # Chatbot interface
    print("\nChatbot is ready! Type 'exit' to end the session.")
    while True:
        query = input("You: ")
        if query.lower() == 'exit':
            break
        response = rag_chain.invoke({"input": query})
        print("Bot:", response["answer"])

if __name__ == "__main__":
    main()

Loading documents from './'...
Processing file: Agri_yield_prediction_imp.csv


libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


Processing file: Copy_of_Groq_API_Usage_Example_(Python).ipynb
Error processing file Copy_of_Groq_API_Usage_Example_(Python).ipynb: Partitioning is not supported for the FileType.UNK file type.
Processing file: CropDataset-Enhanced_imp.csv
Processing file: ICRISAT-District Level Data irri_imp.csv
Processing file: ICRISAT-District Level Data landuse_imp.csv
Processing file: ICRISAT-District Level Data rainmonth_imp.csv
Processing file: ICRISAT-District Level Data seasonfert_imp.csv
Processing file: ICRISAT-District Level Data landuse_imp.csv
Processing file: ICRISAT-District Level Data rainmonth_imp.csv
Processing file: ICRISAT-District Level Data seasonfert_imp.csv
Processing file: ICRISAT-District Level Data soil_imp.csv
Processing file: ICRISAT-District Level Data sourceiir_imp.csv
Processing file: ICRISAT-District Level Data Unapp_imp.csv
Processing file: ICRISAT-District Level Data soil_imp.csv
Processing file: ICRISAT-District Level Data sourceiir_imp.csv
Processing file: ICRISAT-

RuntimeError: Failed to import transformers.models.auto.modeling_auto because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
cannot import name 'ComplexWarning' from 'numpy.core.numeric' (C:\Users\Shrirang\AppData\Roaming\Python\Python311\site-packages\numpy\core\numeric.py)